# Earthquake LSTM Workflow: Steps 9-14

This self-contained notebook continues from `step_8_engineered_features.csv`. It creates the next-month target, performs chronological splits, fits and applies a training-only `RobustScaler`, builds 12-month LSTM windows, and trains a one-month-ahead Keras LSTM model.

## Setup and Input Validation

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display
from sklearn.preprocessing import RobustScaler

INPUT_PATH = Path("data/step_outputs/step_8_engineered_features.csv")
OUTPUT_DIR = Path("data/model_outputs")
MODEL_DIR  = Path("models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

source_df = pd.read_csv(INPUT_PATH)
source_df["month"] = pd.to_datetime(source_df["month"], errors="raise")
source_df = source_df.sort_values("month").reset_index(drop=True)

print(f"Input: {INPUT_PATH.resolve()}")
print(f"Rows: {len(source_df):,}; columns: {len(source_df.columns):,}")
print(f"Date range: {source_df['month'].min().date()} to {source_df['month'].max().date()}")
display(source_df.head())

## Step 9: Create `event_count_next_month`

In [ ]:
FEATURE_COLUMNS = [
    "event_count",
    "mean_magnitude",
    "max_magnitude",
    "mean_depth",
    "tsunami_count",
    "event_count_lag_1",
    "event_count_lag_3",
    "event_count_lag_6",
    "event_count_roll_mean_3",
    "event_count_roll_mean_6",
    "event_count_roll_std_6",
    "event_count_delta_1",
    "max_magnitude_roll_max_6",
    "tsunami_count_roll_sum_12",
    "event_count_outlier_flag",
    "month_sin",
    "month_cos",
]
TARGET_COLUMN = "event_count_next_month"

# CREATE def of STEP 9 BELOW HERE:


rows_before = len(source_df)
target_df = s9_create_target(source_df, FEATURE_COLUMNS, TARGET_COLUMN)

step_9_path = OUTPUT_DIR / "step_9_target_created.csv"
target_df.to_csv(step_9_path, index=False)
print(f"Rows before removing target/warm-up NaNs: {rows_before:,}")
print(f"Complete rows after Step 9: {len(target_df):,}")
print(f"Saved: {step_9_path}")
display(target_df[["month", "event_count", TARGET_COLUMN, "target_month"]].head())
display(target_df[["month", "event_count", TARGET_COLUMN, "target_month"]].tail())

## Step 10: Chronological Train / Validation / Test Split

In [ ]:
TRAIN_END  = pd.Timestamp("2018-12-01")
VAL_START  = pd.Timestamp("2019-01-01")
VAL_END    = pd.Timestamp("2020-12-01")
TEST_START = pd.Timestamp("2021-01-01")

# CREATE def of STEP 10 BELOW HERE:


train_df, val_df, test_df = s10_chronological_split(
    target_df, TRAIN_END, VAL_START, VAL_END, TEST_START
)

split_paths = {
    "train": OUTPUT_DIR / "step_10_train.csv",
    "validation": OUTPUT_DIR / "step_10_validation.csv",
    "test": OUTPUT_DIR / "step_10_test.csv",
}
for split_name, split_df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    split_df.to_csv(split_paths[split_name], index=False)

split_summary = pd.DataFrame(
    [
        {
            "split": name,
            "rows": len(frame),
            "first_target_month": frame["target_month"].min(),
            "last_target_month": frame["target_month"].max(),
        }
        for name, frame in [("train", train_df), ("validation", val_df), ("test", test_df)]
    ]
)
print("Saved Step 10 split CSV files.")
display(split_summary)

## Step 11: Fit `RobustScaler` on Training Features Only

In [ ]:
# CREATE def of STEP 11 BELOW HERE:


feature_scaler, train_scaled_df = s11_fit_scaler(train_df, FEATURE_COLUMNS)

scaler_path = MODEL_DIR / "robust_feature_scaler.joblib"
joblib.dump(feature_scaler, scaler_path)
step_11_path = OUTPUT_DIR / "step_11_train_scaled.csv"
train_scaled_df.to_csv(step_11_path, index=False)

scaler_summary = pd.DataFrame(
    {
        "feature": FEATURE_COLUMNS,
        "center_median": feature_scaler.center_,
        "scale_iqr": feature_scaler.scale_,
    }
)
scaler_summary.to_csv(OUTPUT_DIR / "step_11_scaler_parameters.csv", index=False)
print(f"Scaler fitted on {len(train_df):,} training rows only.")
print(f"Saved scaler: {scaler_path}")
print(f"Saved scaled training data: {step_11_path}")
display(scaler_summary)

## Step 12: Transform Validation and Test Features

In [ ]:
# CREATE def of STEP 12 BELOW HERE:


val_scaled_df, test_scaled_df = s12_transform_splits(val_df, test_df, feature_scaler, FEATURE_COLUMNS)

step_12_val_path = OUTPUT_DIR / "step_12_validation_scaled.csv"
step_12_test_path = OUTPUT_DIR / "step_12_test_scaled.csv"
val_scaled_df.to_csv(step_12_val_path, index=False)
test_scaled_df.to_csv(step_12_test_path, index=False)

all_scaled_df = pd.concat([train_scaled_df, val_scaled_df, test_scaled_df], ignore_index=True)
all_scaled_df = all_scaled_df.sort_values("month").reset_index(drop=True)
if not all_scaled_df["month"].is_monotonic_increasing:
    raise AssertionError("Combined scaled timeline is not chronological.")

print(f"Saved: {step_12_val_path}")
print(f"Saved: {step_12_test_path}")
display(
    pd.DataFrame(
        {
            "split": ["train", "validation", "test"],
            "rows": [len(train_scaled_df), len(val_scaled_df), len(test_scaled_df)],
            "all_features_finite": [
                np.isfinite(train_scaled_df[FEATURE_COLUMNS].to_numpy()).all(),
                np.isfinite(val_scaled_df[FEATURE_COLUMNS].to_numpy()).all(),
                np.isfinite(test_scaled_df[FEATURE_COLUMNS].to_numpy()).all(),
            ],
        }
    )
)

## Step 13: Generate 12-Month LSTM Windows

In [ ]:
# CREATE def of STEP 13 BELOW HERE:


X_all, y_all, window_metadata = create_lstm_windows(
    all_scaled_df, FEATURE_COLUMNS, TARGET_COLUMN, WINDOW_SIZE
)
train_mask = window_metadata["target_month"] <= TRAIN_END
val_mask = window_metadata["target_month"].between(VAL_START, VAL_END)
test_mask = window_metadata["target_month"] >= TEST_START

X_train, y_train = X_all[train_mask], y_all[train_mask]
X_val, y_val = X_all[val_mask], y_all[val_mask]
X_test, y_test = X_all[test_mask], y_all[test_mask]

if X_train.shape[1:] != (WINDOW_SIZE, len(FEATURE_COLUMNS)):
    raise AssertionError(f"Unexpected training window shape: {X_train.shape}")
if not all(np.isfinite(array).all() for array in [X_train, y_train, X_val, y_val, X_test, y_test]):
    raise AssertionError("Window arrays contain NaN or infinite values.")

window_metadata["split"] = np.select(
    [train_mask, val_mask, test_mask],
    ["train", "validation", "test"],
    default="unassigned",
)
windows_path = OUTPUT_DIR / "step_13_lstm_windows.npz"
np.savez_compressed(
    windows_path,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    X_test=X_test,
    y_test=y_test,
    feature_names=np.asarray(FEATURE_COLUMNS),
)
window_metadata_path = OUTPUT_DIR / "step_13_window_metadata.csv"
window_metadata.to_csv(window_metadata_path, index=False)

shape_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "X_shape": [str(X_train.shape), str(X_val.shape), str(X_test.shape)],
        "y_shape": [str(y_train.shape), str(y_val.shape), str(y_test.shape)],
    }
)
print(f"Saved: {windows_path}")
print(f"Saved: {window_metadata_path}")
display(shape_summary)
display(window_metadata.groupby("split")["target_month"].agg(["count", "min", "max"]))

## Step 14: Train One-Month-Ahead LSTM

In [ ]:
# CREATE def of STEP 14.1 BELOW HERE:


# CREATE def of STEP 14.2 BELOW HERE:



model, history = s14_build_and_train_model(
    X_train, y_train, X_val, y_val,
    WINDOW_SIZE, len(FEATURE_COLUMNS),
    LSTM_UNITS, DROPOUT_RATE, DENSE_UNITS,
    LEARNING_RATE, BATCH_SIZE, EPOCHS, PATIENCE,
)

model_path = MODEL_DIR / "earthquake_lstm.keras"
model.save(model_path)
history_df = pd.DataFrame(history.history)
history_df.insert(0, "epoch", np.arange(1, len(history_df) + 1))
history_path = OUTPUT_DIR / "step_14_training_history.csv"
history_df.to_csv(history_path, index=False)

training_summary = pd.DataFrame(
    {
        "epochs_completed": [len(history_df)],
        "best_epoch": [int(history_df["val_loss"].idxmin() + 1)],
        "best_validation_loss": [float(history_df["val_loss"].min())],
        "final_training_loss": [float(history_df["loss"].iloc[-1])],
        "final_validation_loss": [float(history_df["val_loss"].iloc[-1])],
    }
)
print(f"Saved model: {model_path}")
print(f"Saved training history: {history_path}")
display(training_summary)
display(history_df.tail(10))
